# Phase 1: Data Cleaning

In [ ]:
# -------------------------------------------------------------------------
# Step 1: Loading the Excel file and inspect row counts
# -------------------------------------------------------------------------

import numpy as np                                   # Numerical tools
import pandas as pd                                  # Data analysis tools

# File location
file_name = 'Brainpals_sales_analytics.xlsx'        # Main Excel file

# Load every sheet into its own DataFrame
stores = pd.read_excel(file_name, sheet_name='Stores')
products = pd.read_excel(file_name, sheet_name='Products')
weekly_sales = pd.read_excel(file_name, sheet_name='Weekly_Sales')
external = pd.read_excel(file_name, sheet_name='External_Weekly')
customers = pd.read_excel(file_name, sheet_name='Customers')

# Display how many rows and columns each sheet has
print("Stores:        ", stores.shape)
print("Products:      ", products.shape)
print("Weekly Sales:  ", weekly_sales.shape)
print("External:      ", external.shape)
print("Customers:     ", customers.shape)

In [ ]:
# -------------------------------------------------------------------------
# Step 2: Finding duplicates and inconsistent labels
# -------------------------------------------------------------------------

# Check how many exact duplicate rows exist in Weekly_Sales
sales_duplicates = weekly_sales.duplicated().sum()
print("Duplicate rows in Weekly_Sales:", sales_duplicates)

# Inspect category spelling and whitespace in the Products table
print("\nUnique product categories before cleaning:")
print(products['category'].tolist())

In [ ]:
# -------------------------------------------------------------------------
# Step 3: Removing duplicates and cleaning category names
# -------------------------------------------------------------------------

# 1. Drop duplicate rows from Weekly_Sales
weekly_sales = weekly_sales.drop_duplicates()
print("Weekly_Sales rows after dropping duplicates:", len(weekly_sales))

# 2. Clean category names: strip extra spaces and make Title Case
products['category'] = products['category'].str.strip().str.title()
print("\nClean product categories:")
print(products['category'].unique())

In [ ]:
# -------------------------------------------------------------------------
# Step 4: Handling Negative and Missing Sales
# -------------------------------------------------------------------------

# 1. Identify rows with negative or missing sales
invalid_mask = (weekly_sales['units_sold'] <= 0) | (weekly_sales['units_sold'].isnull())
print(f"Total invalid rows to fix: {invalid_mask.sum()}")

# 2. Find median positive sales for each store and product pair
positive_sales = weekly_sales['units_sold'].where(weekly_sales['units_sold'] > 0)
typical_sales = positive_sales.groupby([weekly_sales['store_id'], weekly_sales['product_id']]).transform('median')

# 3. Replace invalid sales with typical sales using .mask()
weekly_sales['units_sold'] = weekly_sales['units_sold'].mask(invalid_mask, typical_sales)

# 4. Confirm sales are clean and positive
print(f"Minimum units_sold after fix: {weekly_sales['units_sold'].min()}")
print(f"Missing units_sold after fix: {weekly_sales['units_sold'].isnull().sum()}")

In [ ]:
# -------------------------------------------------------------------------
# Step 5: Detecting Outliers Using IQR Rule and Per-Product Z-Score
# -------------------------------------------------------------------------

# 1. Calculate IQR fences per product
q25 = weekly_sales.groupby('product_id')['units_sold'].transform('quantile', 0.25)
q75 = weekly_sales.groupby('product_id')['units_sold'].transform('quantile', 0.75)
iqr = q75 - q25

lower_bound = q25 - (1.5 * iqr)
upper_bound = q75 + (1.5 * iqr)

# Flag IQR outliers (outside fences)
weekly_sales['outlier_iqr'] = (weekly_sales['units_sold'] < lower_bound) | (weekly_sales['units_sold'] > upper_bound)

# 2. Calculate Z-scores per product
mean_units = weekly_sales.groupby('product_id')['units_sold'].transform('mean')
std_units = weekly_sales.groupby('product_id')['units_sold'].transform('std')

# Flag Z-score outliers (more than 3 standard deviations away)
weekly_sales['outlier_zscore'] = ((weekly_sales['units_sold'] - mean_units) / std_units).abs() > 3

# 3. Compare results
iqr_total = weekly_sales['outlier_iqr'].sum()
z_total = weekly_sales['outlier_zscore'].sum()
overlap = (weekly_sales['outlier_iqr'] & weekly_sales['outlier_zscore']).sum()

print(f"Rows flagged by IQR rule:         {iqr_total}")
print(f"Rows flagged by Z-score (|z| > 3): {z_total}")
print(f"Rows flagged by both methods:     {overlap}")

In [ ]:
# -------------------------------------------------------------------------
# Step 6: Impute Missing discount_pct from promo_flag and Price Logic
# -------------------------------------------------------------------------

# 1. Count missing values before fix
print(f"Missing discount_pct before: {weekly_sales['discount_pct'].isnull().sum()}")

# 2. Calculate regular undiscounted shelf price
shelf_price = weekly_sales['product_id'].map(products.set_index('product_id')['list_price_ngn']) * (
    weekly_sales['week_start'].map(external.set_index('week_start')['cpi_index']) / 100.0
)

# 3. Calculate implied discount ratio
raw_ratio = 1.0 - (weekly_sales['unit_price_ngn'] / shelf_price)

# 4. Calculate promo discount (snap to nearest 0.05 and clip between 5% and 30%)
promo_disc = (raw_ratio / 0.05).round() * 0.05
promo_disc = promo_disc.clip(lower=0.05, upper=0.30)

# 5. Calculate non-promo discount (0.05 if marked down, else 0.00)
non_promo_disc = pd.Series(0.0, index=weekly_sales.index).mask(raw_ratio >= 0.05, 0.05)

# 6. Select discount based on promo_flag (use promo_disc if promo_flag == 1, else non_promo_disc)
derived_discount = non_promo_disc.mask(weekly_sales['promo_flag'] == 1, promo_disc)

# 7. Fill only the missing entries
weekly_sales['discount_pct'] = weekly_sales['discount_pct'].fillna(derived_discount)

# 8. Verify missing values are resolved
print(f"Missing discount_pct after:  {weekly_sales['discount_pct'].isnull().sum()}")
print("Unique discount values present:", sorted(weekly_sales['discount_pct'].unique().tolist()))

In [ ]:
# -------------------------------------------------------------------------
# Step 7: Cleaning discount_pct Precision & Recomputing revenue_ngn
# -------------------------------------------------------------------------

# 1. Clean float precision jitter on discount_pct
weekly_sales['discount_pct'] = weekly_sales['discount_pct'].round(2)
print("Standardized discount tiers:", sorted(weekly_sales['discount_pct'].unique().tolist()))

# 2. Check mismatches and missing values before recalculation
expected_revenue = weekly_sales['units_sold'] * weekly_sales['unit_price_ngn']
mismatches_before = ((weekly_sales['revenue_ngn'] - expected_revenue).abs() > 0.01).sum()
missing_rev_before = weekly_sales['revenue_ngn'].isnull().sum()

print(f"\nMissing revenue records before:    {missing_rev_before}")
print(f"Mismatched revenue records before: {mismatches_before}")

# 3. Recompute revenue_ngn directly
weekly_sales['revenue_ngn'] = (weekly_sales['units_sold'] * weekly_sales['unit_price_ngn']).round(2)

# 4. Verify mathematical integrity across all 46,800 rows
mismatches_after = ((weekly_sales['revenue_ngn'] - (weekly_sales['units_sold'] * weekly_sales['unit_price_ngn'])).abs() > 0.01).sum()
missing_rev_after = weekly_sales['revenue_ngn'].isnull().sum()

print(f"\nMissing revenue records after:     {missing_rev_after}")
print(f"Mismatched revenue records after:  {mismatches_after}")
print(f"Revenue range: NGN {weekly_sales['revenue_ngn'].min():,.2f} to NGN {weekly_sales['revenue_ngn'].max():,.2f}")

In [ ]:
# -------------------------------------------------------------------------
# Step 8: Impute Store Foot Traffic & Clean Customer Age Anomalies
# -------------------------------------------------------------------------

# 1. Impute missing foot_traffic_index in Stores using the store_type median
missing_traffic_before = stores['foot_traffic_index'].isnull().sum()
print(f"Stores with missing foot_traffic_index before: {missing_traffic_before}")

# Calculate median foot traffic for each store format (Express, Standard, Hypermarket)
type_traffic_median = stores.groupby('store_type')['foot_traffic_index'].transform('median')

# Fill missing values using the store_type median
stores['foot_traffic_index'] = stores['foot_traffic_index'].fillna(type_traffic_median)

missing_traffic_after = stores['foot_traffic_index'].isnull().sum()
print(f"Stores with missing foot_traffic_index after:  {missing_traffic_after}")

# 2. Flag impossible customer ages vs missing customer ages
customers['invalid_age_flag'] = (customers['age'] <= 0) | (customers['age'] > 100)
customers['missing_age_flag'] = customers['age'].isnull()

print(f"\nImpossible customer ages flagged (<= 0 or > 100): {customers['invalid_age_flag'].sum()}")
print(f"Missing customer ages flagged (NaN):               {customers['missing_age_flag'].sum()}")

# Set impossible ages to None so they do not distort demographic calculations
customers.loc[customers['invalid_age_flag'], 'age'] = None

# Check valid age summary statistics
valid_ages = customers['age'].dropna()
print(f"\nClean Customer Age Range: {valid_ages.min():.0f} to {valid_ages.max():.0f} years (Median: {valid_ages.median():.0f})")

In [ ]:
# -------------------------------------------------------------------------
# Step 9: Lossless Table Merge (Weekly_Sales + Stores + Products + External)
# -------------------------------------------------------------------------

# 1. Capture initial row count
initial_rows = len(weekly_sales)

# 2. Sequential Left Joins
master_df = weekly_sales.merge(stores, on='store_id', how='left')
master_df = master_df.merge(products, on='product_id', how='left')
master_df = master_df.merge(external, on='week_start', how='left')

# 3. Verify row counts and column totals
final_rows = len(master_df)

print(f"Weekly sales rows before merge:  {initial_rows:,}")
print(f"Master dataset rows after merge: {final_rows:,}")
print(f"Total merged columns:             {master_df.shape[1]}")

# 4. Integrity assertion check
assert initial_rows == final_rows == 46800, "Row count mismatch: Data was dropped or duplicated during merge!"
print("Verification check passed: Exactly 46,800 panel rows preserved.")

# 5. Check if any missing values exist in the merged dataset
remaining_nulls = master_df.isnull().sum().sum()
print(f"Total remaining null values across master table: {remaining_nulls}")

In [ ]:
# Just checking first five rows of the merged columns
master_df.head()

# Phase 2: Feature Engineering

In [ ]:
import math

# -------------------------------------------------------------------------
# Step 10: Real Prices, Price Ratios, and Log Transformations
# -------------------------------------------------------------------------

# 1. Deflate nominal prices using CPI index (base 100)
master_df['real_price'] = (master_df['unit_price_ngn'] / master_df['cpi_index']) * 100.0

# 2. Base period real list price equals list_price_ngn
master_df['real_list_price'] = master_df['list_price_ngn']

# 3. Compute price ratio
master_df['price_ratio'] = master_df['real_price'] / master_df['real_list_price']

# 4. Compute natural logarithms using math.log
master_df['log_units'] = master_df['units_sold'].apply(math.log)
master_df['log_price_ratio'] = master_df['price_ratio'].apply(math.log)

# Summary check
print("Feature Engineering Check (Prices & Logs):")
print(f"Price Ratio range: {master_df['price_ratio'].min():.2f} to {master_df['price_ratio'].max():.2f}")
print(f"Log Units range:   {master_df['log_units'].min():.2f} to {master_df['log_units'].max():.2f}")

In [ ]:
# -------------------------------------------------------------------------
# Step 11: Calendar Features and Trigonometric Seasonal Terms
# -------------------------------------------------------------------------

# 1. Ensure week_start is datetime format
master_df['week_start'] = pd.to_datetime(master_df['week_start'])

# 2. Extract standard calendar parts
master_df['week_of_year'] = master_df['week_start'].dt.isocalendar().week.astype(int)  # 1 to 52
master_df['month'] = master_df['week_start'].dt.month                                   # 1 to 12
master_df['quarter'] = master_df['week_start'].dt.quarter                               # 1 to 4

# 3. Create cyclical Fourier terms using Python's built-in math module
master_df['sin_season'] = master_df['week_of_year'].apply(lambda w: math.sin(2 * math.pi * w / 52.0))
master_df['cos_season'] = master_df['week_of_year'].apply(lambda w: math.cos(2 * math.pi * w / 52.0))

# Check summary ranges
print("Calendar & Seasonality Features:")
print(f"Week of year range: {master_df['week_of_year'].min()} to {master_df['week_of_year'].max()}")
print(f"Month range:        {master_df['month'].min()} to {master_df['month'].max()}")
print(f"Quarter range:      {master_df['quarter'].min()} to {master_df['quarter'].max()}")
print(f"Sin term range:     {master_df['sin_season'].min():.2f} to {master_df['sin_season'].max():.2f}")
print(f"Cos term range:     {master_df['cos_season'].min():.2f} to {master_df['cos_season'].max():.2f}")

In [ ]:
# -------------------------------------------------------------------------
# Step 12: Profit Margins & Store Productivity Metrics
# -------------------------------------------------------------------------

# 1. Total Cost and Gross Profit in NGN
total_cost = master_df['units_sold'] * master_df['unit_cost_ngn']
master_df['gross_profit'] = (master_df['revenue_ngn'] - total_cost).round(2)

# 2. Profit Margin Percentage (Gross Profit ÷ Revenue)
master_df['margin_pct'] = (master_df['gross_profit'] / master_df['revenue_ngn']).round(4)

# 3. Total store revenue over the entire period (aggregated by store)
store_total_rev = master_df.groupby('store_id')['revenue_ngn'].transform('sum')

# 4. Store Productivity metrics
master_df['sales_per_sqm'] = (store_total_rev / master_df['size_sqm']).round(2)
master_df['staff_productivity'] = (store_total_rev / master_df['staff_count']).round(2)

# Verification check
print("Margins and Productivity calculated:")
print(f"Mean Gross Margin:          {master_df['margin_pct'].mean() * 100:.2f}%")
print(f"Sales per sqm (range):      NGN {master_df['sales_per_sqm'].min():,.2f} to NGN {master_df['sales_per_sqm'].max():,.2f}")
print(f"Staff productivity (range): NGN {master_df['staff_productivity'].min():,.2f} to NGN {master_df['staff_productivity'].max():,.2f}")

In [ ]:
# -------------------------------------------------------------------------
# Step 13: Time-Series 4-Week Lags and Rolling Means
# -------------------------------------------------------------------------

# 1. Sort chronologically by store, product, and week
master_df = master_df.sort_values(['store_id', 'product_id', 'week_start']).reset_index(drop=True)

# 2. Group by store and product pair
pair_group = master_df.groupby(['store_id', 'product_id'])['units_sold']

# 3. 4-week lag (sales volume from 4 weeks prior)
master_df['units_lag_4'] = pair_group.shift(4)

# 4. 4-week rolling mean
master_df['units_rolling_4'] = pair_group.transform(lambda s: s.rolling(4).mean())

# 5. Handle initial 4-week boundary nulls within each group
master_df['units_lag_4'] = master_df.groupby(['store_id', 'product_id'])['units_lag_4'].bfill()
master_df['units_rolling_4'] = master_df.groupby(['store_id', 'product_id'])['units_rolling_4'].bfill()

print("Time-series features created successfully:")
print(f"Missing lag-4 units:        {master_df['units_lag_4'].isnull().sum()}")
print(f"Missing rolling-4 units:    {master_df['units_rolling_4'].isnull().sum()}")

In [ ]:
# -------------------------------------------------------------------------
# Step 14: Save Cleaned Master Table to CSV
# -------------------------------------------------------------------------

# Export to CSV without index column
master_df.to_csv('cleaned_master.csv', index=False)

print(f"cleaned_master.csv created successfully!")
print(f"Final shape: {master_df.shape[0]:,} rows and {master_df.shape[1]} columns.")